# 📚 Verificador de Citações Acadêmicas

**Como usar — siga os passos em ordem:**
1. Clique no botão ▶ de cada célula, de cima para baixo
2. Aguarde cada célula terminar (o ▶ vira ✅) antes de passar para a próxima
3. Leia as instruções que aparecem abaixo de cada célula

> 💡 **Dica:** Os arquivos de referência precisam estar no seu Google Drive.
> No tablet, abra o app **Arquivos**, copie a pasta do pen drive e cole no **Google Drive**.

---
## ⚙️ PASSO 1 — Preparar o ambiente
_Rode esta célula uma única vez. Pode demorar 1-2 minutos._

In [ ]:
print('Instalando bibliotecas...')
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'anthropic', 'pymupdf', '-q'],
               capture_output=True)

print('Baixando o código do verificador...')
subprocess.run(['curl', '-sL',
    'https://raw.githubusercontent.com/tenentevilaca/dissertacao/claude/citation-verification-tool-ZAYFh/desktop/analisar.py',
    '-o', 'analisar.py'], capture_output=True)

import pathlib
if pathlib.Path('analisar.py').stat().st_size > 1000:
    print('\n✅ Tudo pronto! Vá para o Passo 2.')
else:
    print('\n❌ Erro ao baixar o código. Verifique sua conexão e rode novamente.')

---
## 📁 PASSO 2 — Conectar ao Google Drive
_O Google Drive é onde estão os seus PDFs de referência.
Vai aparecer um link para autorizar — clique nele e cole o código._

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('\n✅ Google Drive conectado!')
print('Seus arquivos estão em: /content/drive/MyDrive/')
print('\nPasta raiz do Drive:')
import os
itens = [f for f in os.listdir('/content/drive/MyDrive') 
         if not f.startswith('.')]
for item in sorted(itens)[:20]:
    tipo = '📁' if os.path.isdir(f'/content/drive/MyDrive/{item}') else '📄'
    print(f'  {tipo} {item}')

---
## 📄 PASSO 3 — Enviar a dissertação
_Clique em ▶ e depois no botão **Escolher arquivos** que vai aparecer.
Selecione o arquivo **.docx** da sua dissertação._

In [ ]:
from google.colab import files as _colab_files
print('👇 Selecione o arquivo .docx da dissertação:')
uploaded = _colab_files.upload()

if uploaded:
    DISS_PATH = list(uploaded.keys())[0]
    tamanho = len(list(uploaded.values())[0]) // 1024
    print(f'\n✅ Arquivo recebido: {DISS_PATH} ({tamanho} KB)')
else:
    print('\n❌ Nenhum arquivo selecionado. Rode esta célula novamente.')

---
## 🗂️ PASSO 4 — Informar a pasta de referências
_Informe o caminho da pasta no Google Drive que contém os PDFs._

**Exemplo:** se a sua pasta se chama `fusions centers` e está na raiz do Drive,
o caminho é: `/content/drive/MyDrive/fusions centers`

In [ ]:
import os

print('Pastas disponíveis no Drive (nível 1):')
drive_root = '/content/drive/MyDrive'
pastas = sorted([f for f in os.listdir(drive_root)
                 if os.path.isdir(os.path.join(drive_root, f))
                 and not f.startswith('.')])
for p in pastas:
    n = len([x for x in os.listdir(os.path.join(drive_root, p))])
    print(f'  📁 {p}  ({n} itens)')

print()
REFS_DIR = input('▶ Digite o caminho completo da pasta de referências:\n'
                 '  (ex: /content/drive/MyDrive/fusions centers)\n  → ')

REFS_DIR = REFS_DIR.strip().strip('"').strip("'")
if os.path.isdir(REFS_DIR):
    arqs = [f for f in os.listdir(REFS_DIR)
            if f.lower().endswith(('.pdf','.docx','.doc','.txt'))]
    print(f'\n✅ Pasta encontrada! {len(arqs)} arquivo(s) de referência.')
else:
    print(f'\n❌ Pasta não encontrada: {REFS_DIR}')
    print('   Verifique o caminho e rode esta célula novamente.')

---
## 🔑 PASSO 5 — Chave API (opcional)
_Necessária apenas para a verificação semântica com IA.
Se não tiver, deixe em branco e pressione Enter — o sistema fará só o cruzamento._

In [ ]:
import getpass
API_KEY = getpass.getpass('🔑 Cole sua chave API Claude (sk-ant-...) e pressione Enter\n'
                          '   (ou pressione Enter sem digitar nada para pular): ')
if API_KEY.strip():
    print('✅ Chave API recebida (verificação semântica ativada).')
else:
    print('ℹ️  Sem chave API — só será feito o cruzamento de citações e referências.')

---
## 🚀 PASSO 6 — Rodar a análise
_Esta é a etapa principal. O log vai aparecer aqui embaixo em tempo real.
Pode demorar vários minutos dependendo do número de arquivos._

In [ ]:
import importlib, sys, analisar
importlib.reload(analisar)  # garante versão atualizada

from IPython.display import clear_output
import time

_log_buffer = []

def _log(msg):
    _log_buffer.append(msg)
    # Mostra as últimas 40 linhas para não sobrecarregar
    clear_output(wait=True)
    print('\n'.join(_log_buffer[-60:]))

print('▶ Iniciando análise...')
RESULTADO = analisar.analisar(
    diss_path=DISS_PATH,
    refs_dir=REFS_DIR,
    api_key=API_KEY,
    log_fn=_log,
)

if RESULTADO:
    print('\n' + '='*60)
    print('✅ ANÁLISE CONCLUÍDA! Vá para o Passo 7 para ver o relatório.')
else:
    print('\n❌ A análise não produziu resultado. Verifique o log acima.')

---
## 📊 PASSO 7 — Ver e salvar o relatório
_O relatório vai aparecer aqui e também será baixado automaticamente._

In [ ]:
if not RESULTADO:
    print('❌ Sem resultado para mostrar. Complete o Passo 6 primeiro.')
else:
    import analisar
    from IPython.display import HTML, display
    from google.colab import files as _cf
    from datetime import datetime

    html_content = analisar.gerar_html(RESULTADO)

    nome_arquivo = f'relatorio_citacoes_{datetime.now().strftime("%Y%m%d_%H%M")}.html'
    with open(nome_arquivo, 'w', encoding='utf-8') as f:
        f.write(html_content)

    # Salva também no Drive
    import shutil
    destino_drive = f'/content/drive/MyDrive/{nome_arquivo}'
    shutil.copy(nome_arquivo, destino_drive)

    print(f'✅ Relatório salvo no Google Drive: {nome_arquivo}')
    print('\n📥 Iniciando download para o tablet...')
    _cf.download(nome_arquivo)

    print('\n📊 Prévia do relatório:')
    display(HTML(html_content))